# Lightweight multimodal fashion attribute extraction — training & evaluation
Gated late fusion (ConvNeXt-Tiny + DeBERTa-v3 Small), keyword masking against label leakage, category-conditional masked loss.

**Run order:** top to bottom. Set `DATA_DIR` in the config cell. Images and raw product text are **not** redistributed (Kaggle *Fashion Product Images* dataset); the processed CSV used here (`Research final dataset.csv`, containing the derived attribute labels) is not included. Released artefacts: `train_split.csv`, `val_split.csv`, `test_split.csv`, `masking.py`.

In [ ]:
!pip install -q timm transformers scikit-learn pandas Pillow scipy statsmodels

In [ ]:
import os, re, json, random, time, pickle, gc
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ---- EDIT THESE PATHS ----
DATA_DIR   = "/path/to/Dataset"                       # folder containing the CSV and images_final/
CSV_PATH   = f"{DATA_DIR}/Research final dataset.csv"
IMAGES_DIR = f"{DATA_DIR}/images_final"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED          = 42
MAX_TOKEN_LEN = 128
IMG_SIZE      = 224
BATCH_IMAGE   = 32
BATCH_TEXT    = 16
BATCH_MULTI   = 6          # multimodal batch size (memory-limited, Kaggle T4)
LR_BACKBONE   = 1e-5
LR_HEAD       = 1e-4
LR_ENCODER_MM = 5e-6
WEIGHT_DECAY  = 1e-2
DROPOUT       = 0.3
MAX_EPOCHS    = 20
PATIENCE      = 5

TARGET_ATTRIBUTES = [
    'category', 'color', 'pattern', 'material',
    'sleeve_type', 'fit', 'neck_type',
    'heel_type', 'toe_style', 'sole_type',
    'gender', 'occasion', 'closure_type'
]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

## 1. Keyword masking (identical to `masking.py`)

In [ ]:
ATTR_KEYWORDS = {
    'color':   ['black','blue','brown','gold','green','grey','gray','orange','pink',
                'purple','red','silver','white','yellow','navy','beige','cream',
                'maroon','olive','khaki','teal','coral','turquoise','mustard','charcoal'],
    'pattern': ['checked','check','embroidered','embroidery','floral','graphic',
                'polka dot','polka-dot','printed','print','solid','striped',
                'stripe','stripes'],
    'material':['canvas','cotton','denim','eva','fabric','fleece','leather','linen',
                'lycra','metal','polyester','rayon','rubber','silk','synthetic',
                'tpr','textile','wool','nylon','suede','velvet','mesh','knit'],
    'sleeve':  ['full sleeve','half sleeve','sleeveless','three-quarter sleeve',
                'long sleeve','short sleeve','short sleeves','long sleeves',
                'full sleeves','half sleeves','mid sleeve'],
    'fit':     ['flared fit','loose fit','regular fit','slim fit','straight fit',
                'regular wash','slim wash','fits regular','fits slim',
                'size and fit','hem fit','hemline fit','chest fit','front fit',
                'front fitting','hem fitting','hemline fitting','fitting'],
    'neck':    ['boat neck','off shoulder','round neck','square neck','turtle neck',
                'v-neck','v neck','scoop neck','mock collar','polo neck',
                'polo collar','spread collar','tipped collar','ribbed collar',
                'mandarin collar','collar silhouette','round neckline',
                'ribbed round neckline','neckline'],
    'closure': ['pull-on','lace-up','slip-on','velcro',
                'buckle closure','elastic closure','clasp closure',
                'zipper closure','button closure','butterfly clasp',
                'fold over clasp','buttoned cuffs','buttoned front',
                'buttoned placket','buttoned chest','buttoned flaps',
                'full buttoned','button placket','button cuffs',
                'zippered fly','zipper fly','drawstrings',
                'elasticated waist','elasticated waistband',
                'with zippers','with buttons','zipper and button',
                'gunmetal buckle','single button','double button',
                'pearlescent white buttons','drawstring'],
    'occasion':['casual','formal','sports','festive','party','beach',
                'lounge','outdoor','wedding','daily use'],
    'gender':  ['women','woman','womens','womans','lady','ladies','mens','unisex',
                'girls','girl','boys','boy','men','man','mans'],
}

def mask_text(text):
    t = str(text).lower()
    for attr, kws in ATTR_KEYWORDS.items():
        for kw in sorted(kws, key=len, reverse=True):
            t = re.sub(r'\b' + re.escape(kw) + r'\b',
                       f'[{attr.upper()}]', t)
    return t

## 2. Data, masking, label encoding, split

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {len(df)} rows")

# sanity check: every referenced image must exist
missing = [f for f in df['image'].astype(str) if not os.path.exists(os.path.join(IMAGES_DIR, f))]
print("Missing images:", len(missing))
assert len(missing) == 0, "Image files not found - check IMAGES_DIR"

df['masked_text'] = df['text'].apply(mask_text)

# Leakage proxy: case-insensitive SUBSTRING match of the label value in the text
print("\nLeakage proxy (before -> after masking):")
check_cols = ['color','pattern','material','sleeve_type','fit','neck_type','gender','occasion','closure_type']
for col in check_cols:
    if col not in df.columns:
        continue
    valid = df[~df[col].str.lower().isin(['none','unknown',''])]
    if len(valid) == 0:
        continue
    before = sum(1 for _, r in valid.iterrows() if str(r[col]).lower() in str(r['text']).lower())
    after  = sum(1 for _, r in valid.iterrows() if str(r[col]).lower() in str(r['masked_text']).lower())
    print(f"  {col:15s}: {before/len(valid)*100:5.1f}% -> {after/len(valid)*100:5.1f}%")

for attr in TARGET_ATTRIBUTES:
    df[attr] = df[attr].fillna('Unknown').replace({'none': 'Unknown', 'None': 'Unknown'})

label_encoders, num_classes = {}, {}
for attr in TARGET_ATTRIBUTES:
    le = LabelEncoder()
    df[attr + '_enc'] = le.fit_transform(df[attr])
    label_encoders[attr] = le
    num_classes[attr] = len(le.classes_)
    print(f"  {attr:15s}: {num_classes[attr]} classes")

with open(f"{OUTPUT_DIR}/label_encoders.pkl", 'wb') as f:
    pickle.dump(label_encoders, f)
with open(f"{OUTPUT_DIR}/num_classes.json", 'w') as f:
    json.dump(num_classes, f)

# 70 / 15 / 15 split, stratified by category, seed 42
train_df, temp = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['category_enc'])
val_df, test_df = train_test_split(temp, test_size=0.50, random_state=SEED, stratify=temp['category_enc'])

# Release only the label/split files (no raw text, no images)
release_cols = ['image','brand','category','color','pattern','material','sleeve_type','fit',
                'neck_type','heel_type','toe_style','sole_type','gender','occasion','closure_type']
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d[release_cols].to_csv(f"{OUTPUT_DIR}/{name}_split.csv", index=False)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 3. Datasets

In [ ]:
train_tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class ImgDS(Dataset):
    def __init__(self, df, tfm):
        self.df  = df.reset_index(drop=True)
        self.tfm = tfm
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            img = Image.open(
                os.path.join(IMAGES_DIR, str(r['image']))).convert('RGB')
        except (FileNotFoundError, OSError):
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128,128,128))
        return self.tfm(img), {a: int(r[a+'_enc']) for a in TARGET_ATTRIBUTES}

class TxtDS(Dataset):
    def __init__(self, df, tok):
        self.df  = df.reset_index(drop=True)
        self.tok = tok
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        e = self.tok(str(r['masked_text']),
                     max_length=MAX_TOKEN_LEN,
                     padding='max_length',
                     truncation=True,
                     return_tensors='pt')
        return {'input_ids':      e['input_ids'].squeeze(),
                'attention_mask': e['attention_mask'].squeeze()}, \
               {a: int(r[a+'_enc']) for a in TARGET_ATTRIBUTES}

class MMDS(Dataset):
    def __init__(self, df, tok, tfm):
        self.df  = df.reset_index(drop=True)
        self.tok = tok
        self.tfm = tfm
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            img = Image.open(
                os.path.join(IMAGES_DIR, str(r['image']))).convert('RGB')
        except (FileNotFoundError, OSError):
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128,128,128))
        img = self.tfm(img)
        e = self.tok(str(r['masked_text']),
                     max_length=MAX_TOKEN_LEN,
                     padding='max_length',
                     truncation=True,
                     return_tensors='pt')
        return img, \
               {'input_ids':      e['input_ids'].squeeze(),
                'attention_mask': e['attention_mask'].squeeze()}, \
               {a: int(r[a+'_enc']) for a in TARGET_ATTRIBUTES}


tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
print("Datasets ready.")

## 4. Models

In [ ]:
class ImageModel(nn.Module):
    def __init__(self, nc):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_tiny', pretrained=True, num_classes=0)
        d = self.backbone.num_features
        self.proj  = nn.Sequential(
            nn.Linear(d, 512), nn.LayerNorm(512), nn.ReLU())
        self.drop  = nn.Dropout(DROPOUT)
        self.heads = nn.ModuleDict(
            {a: nn.Linear(512, n) for a, n in nc.items()})
    def forward(self, x):
        x = self.drop(self.proj(self.backbone(x)))
        return {a: h(x) for a, h in self.heads.items()}

class TextModel(nn.Module):
    def __init__(self, nc, name='microsoft/deberta-v3-small'):
        super().__init__()
        self.enc   = AutoModel.from_pretrained(name)
        d          = self.enc.config.hidden_size
        self.drop  = nn.Dropout(DROPOUT)
        self.heads = nn.ModuleDict(
            {a: nn.Linear(d, n) for a, n in nc.items()})
    def forward(self, ids, mask):
        out = self.enc(input_ids=ids, attention_mask=mask)
        cls = out.last_hidden_state[:, 0, :].float()  # fix dtype for Kaggle T4 x2
        cls = self.drop(cls)
        return {a: h(cls) for a, h in self.heads.items()}

class MultimodalModel(nn.Module):
    def __init__(self, nc,
                 img='convnext_tiny',
                 txt='microsoft/deberta-v3-small'):
        super().__init__()
        self.backbone = timm.create_model(
            img, pretrained=True, num_classes=0)
        self.enc      = AutoModel.from_pretrained(txt)
        di = self.backbone.num_features
        dt = self.enc.config.hidden_size
        self.ip   = nn.Sequential(
            nn.Linear(di, 512), nn.LayerNorm(512), nn.ReLU())
        self.tp   = nn.Sequential(
            nn.Linear(dt, 512), nn.LayerNorm(512), nn.ReLU())
        self.gate = nn.Sequential(
            nn.Linear(1024, 64), nn.ReLU(),
            nn.Linear(64, 1),   nn.Sigmoid())
        self.drop  = nn.Dropout(DROPOUT)
        self.heads = nn.ModuleDict(
            {a: nn.Linear(512, n) for a, n in nc.items()})
    def forward(self, img, ids, mask):
        iv    = self.ip(self.backbone(img)).float()  # fix dtype
        tv    = self.tp(
            self.enc(input_ids=ids,
                     attention_mask=mask
                     ).last_hidden_state[:, 0, :].float())  # fix dtype
        alpha = self.gate(torch.cat([iv, tv], dim=1))
        f     = self.drop(alpha * iv + (1 - alpha) * tv)
        return {a: h(f) for a, h in self.heads.items()}, alpha

# record which ConvNeXt checkpoint timm actually loads
_tmp = timm.create_model('convnext_tiny', pretrained=True, num_classes=0)
print("ConvNeXt pretrained_cfg:", _tmp.pretrained_cfg)
del _tmp

## 5. Category-conditional masked loss and evaluation

In [ ]:
ATTR_CATEGORY_MAP = {
    'sleeve_type': ['Dresses','Innerwear Vests','Jackets','Kurtas','Kurtis',
                    'Night suits','Nightdress','Shirts','Sweaters',
                    'Sweatshirts','Tops','Tshirts','Tunics'],
    'neck_type':   ['Dresses','Innerwear Vests','Jackets','Kurtas','Kurtis',
                    'Night suits','Nightdress','Shirts','Sweaters',
                    'Sweatshirts','Tops','Tshirts','Tunics'],
    'fit':         ['Capris','Dresses','Innerwear Vests','Jackets','Jeans',
                    'Kurtas','Kurtis','Leggings','Night suits','Nightdress',
                    'Shirts','Shorts','Skirts','Sweaters','Sweatshirts',
                    'Tops','Track Pants','Trousers','Tshirts','Tunics'],
    'heel_type':   ['Casual Shoes','Flats','Flip Flops','Formal Shoes',
                    'Heels','Sandals','Sports Shoes'],
    'toe_style':   ['Casual Shoes','Flats','Flip Flops','Formal Shoes',
                    'Heels','Sandals','Sports Shoes'],
    'sole_type':   ['Casual Shoes','Flats','Flip Flops','Formal Shoes',
                    'Heels','Sandals','Sports Shoes'],
}

ATTR_VALID_INDICES = {}
for attr, cats in ATTR_CATEGORY_MAP.items():
    ATTR_VALID_INDICES[attr] = set(
        label_encoders['category'].transform([c])[0]
        for c in cats
        if c in list(label_encoders['category'].classes_)
    )

crit_none = nn.CrossEntropyLoss(reduction='none')
crit_mean = nn.CrossEntropyLoss(reduction='mean')

def mloss(out, lbl):
    total = 0.0
    count = 0
    cat_labels = lbl['category']
    for a in TARGET_ATTRIBUTES:
        pred = out[a]
        true = lbl[a].to(DEVICE)
        if a in ATTR_VALID_INDICES:
            valid_set = ATTR_VALID_INDICES[a]
            mask = torch.tensor(
                [1.0 if c.item() in valid_set else 0.0
                 for c in cat_labels],
                device=DEVICE)
            if mask.sum() > 0:
                per_sample = crit_none(pred, true)
                total += (per_sample * mask).sum() / mask.sum()
                count += 1
        else:
            total += crit_mean(pred, true)
            count += 1
    return total / count if count > 0 else torch.tensor(0.0, device=DEVICE)

def filter_by_category(preds, truths, cats, attr):
    valid = ATTR_VALID_INDICES.get(attr, None)
    if valid is None:
        return preds, truths
    fp = [preds[i] for i,c in enumerate(cats) if c in valid]
    ft = [truths[i] for i,c in enumerate(cats) if c in valid]
    return fp, ft

def eval_img(m, dl):
    m.eval()
    P = {a: [] for a in TARGET_ATTRIBUTES}
    T = {a: [] for a in TARGET_ATTRIBUTES}
    C = []
    with torch.no_grad():
        for x, l in dl:
            o = m(x.to(DEVICE))
            C += l['category'].tolist()
            for a in TARGET_ATTRIBUTES:
                P[a] += o[a].argmax(1).cpu().tolist()
                T[a] += l[a].tolist()
    acc = {}
    for a in TARGET_ATTRIBUTES:
        fp, ft = filter_by_category(P[a], T[a], C, a)
        acc[a] = accuracy_score(ft, fp) if ft else 0.0
    return acc, np.mean(list(acc.values())), P, T

def eval_txt(m, dl):
    m.eval()
    P = {a: [] for a in TARGET_ATTRIBUTES}
    T = {a: [] for a in TARGET_ATTRIBUTES}
    C = []
    with torch.no_grad():
        for x, l in dl:
            o = m(x['input_ids'].to(DEVICE),
                  x['attention_mask'].to(DEVICE))
            C += l['category'].tolist()
            for a in TARGET_ATTRIBUTES:
                P[a] += o[a].argmax(1).cpu().tolist()
                T[a] += l[a].tolist()
    acc = {}
    for a in TARGET_ATTRIBUTES:
        fp, ft = filter_by_category(P[a], T[a], C, a)
        acc[a] = accuracy_score(ft, fp) if ft else 0.0
    return acc, np.mean(list(acc.values())), P, T

def eval_mm(m, dl):
    m.eval()
    P = {a: [] for a in TARGET_ATTRIBUTES}
    T = {a: [] for a in TARGET_ATTRIBUTES}
    C = []
    A = []
    with torch.no_grad():
        for img, x, l in dl:
            o, al = m(img.to(DEVICE),
                      x['input_ids'].to(DEVICE),
                      x['attention_mask'].to(DEVICE))
            A += al.squeeze(1).cpu().tolist()
            C += l['category'].tolist()
            for a in TARGET_ATTRIBUTES:
                P[a] += o[a].argmax(1).cpu().tolist()
                T[a] += l[a].tolist()
    acc = {}
    for a in TARGET_ATTRIBUTES:
        fp, ft = filter_by_category(P[a], T[a], C, a)
        acc[a] = accuracy_score(ft, fp) if ft else 0.0
    return acc, np.mean(list(acc.values())), A, P, T

## 6. Train image-only model (5 warm-up epochs with frozen backbone, then full fine-tuning)

In [ ]:
print("="*50)
print("TRAINING IMAGE-ONLY MODEL")
print("="*50)

tr_dl = DataLoader(ImgDS(train_df, train_tfm),
                   BATCH_IMAGE, shuffle=True,
                   num_workers=2, pin_memory=True)
vl_dl = DataLoader(ImgDS(val_df, val_tfm),
                   BATCH_IMAGE, shuffle=False,
                   num_workers=2, pin_memory=True)
ts_dl = DataLoader(ImgDS(test_df, val_tfm),
                   BATCH_IMAGE, shuffle=False,
                   num_workers=2, pin_memory=True)

img_m = ImageModel(num_classes).to(DEVICE)

# ── Warmup: backbone frozen ───────────────────────────────────
for p in img_m.backbone.parameters():
    p.requires_grad = False

o1 = optim.AdamW(
    filter(lambda p: p.requires_grad, img_m.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

for ep in range(5):
    img_m.train(); tl = 0
    for x, l in tr_dl:
        o1.zero_grad()
        loss = mloss(img_m(x.to(DEVICE)), l)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            img_m.parameters(), 1.0)
        o1.step()
        tl += loss.item()
    _, va, _, _ = eval_img(img_m, vl_dl)
    print(f"  Warmup {ep+1}/5 | "
          f"Loss: {tl/len(tr_dl):.4f} | "
          f"Val: {va*100:.2f}%")

# ── Full fine-tuning: backbone unfrozen ───────────────────────
for p in img_m.backbone.parameters():
    p.requires_grad = True

o2 = optim.AdamW([
    {'params': img_m.backbone.parameters(),
     'lr': LR_BACKBONE},
    {'params': list(img_m.proj.parameters()) +
               list(img_m.heads.parameters()),
     'lr': LR_HEAD}
], weight_decay=WEIGHT_DECAY)

sc = optim.lr_scheduler.CosineAnnealingLR(
    o2, T_max=MAX_EPOCHS, eta_min=1e-7)

# early stopping / checkpoint selection on validation accuracy
best = 0.0; pc = 0
img_tl = []; img_va = []

for ep in range(MAX_EPOCHS):
    img_m.train(); tl = 0
    for x, l in tr_dl:
        o2.zero_grad()
        loss = mloss(img_m(x.to(DEVICE)), l)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            img_m.parameters(), 1.0)
        o2.step()
        tl += loss.item()
    sc.step()
    avg = tl / len(tr_dl)
    _, va, _, _ = eval_img(img_m, vl_dl)
    img_tl.append(avg); img_va.append(va)
    print(f"  Epoch {ep+1:2d}/{MAX_EPOCHS} | "
          f"Loss: {avg:.4f} | Val: {va*100:.2f}%")

    # keep the checkpoint with best VALIDATION accuracy
    if va > best:
        best = va; pc = 0
        torch.save(img_m.state_dict(),
                   f"{OUTPUT_DIR}/img_best.pth")
        print(f"  ✓ Best saved ({va*100:.2f}%)")
    else:
        pc += 1
        if pc >= PATIENCE:
            print(f"  Early stop at epoch {ep+1}")
            break

print("Image model training complete.")

# ── Test evaluation ───────────────────────────────────────────
img_m.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/img_best.pth"))

img_accs, img_overall, img_preds, img_truths = \
    eval_img(img_m, ts_dl)

print(f"\nImage-Only TEST Accuracy: {img_overall*100:.2f}%")
print("Per-attribute:")
for a, acc in img_accs.items():
    print(f"  {a:15s}: {acc*100:.2f}%")

## 7. Train text-only model

In [ ]:
print("="*50)
print("TRAINING TEXT-ONLY MODEL")
print("="*50)

tr_dl = DataLoader(TxtDS(train_df, tokenizer),
                   BATCH_TEXT, shuffle=True,
                   num_workers=2, pin_memory=True)
vl_dl = DataLoader(TxtDS(val_df, tokenizer),
                   BATCH_TEXT, shuffle=False,
                   num_workers=2, pin_memory=True)
ts_dl = DataLoader(TxtDS(test_df, tokenizer),
                   BATCH_TEXT, shuffle=False,
                   num_workers=2, pin_memory=True)

txt_m = TextModel(num_classes).to(DEVICE)

# Fix dtype mismatch on Kaggle T4 x2
txt_m = txt_m.float()
for param in txt_m.parameters():
    param.data = param.data.float()

ot = optim.AdamW([
    {'params': txt_m.enc.parameters(),   'lr': LR_BACKBONE},
    {'params': txt_m.heads.parameters(), 'lr': LR_HEAD}
], weight_decay=WEIGHT_DECAY)

st = optim.lr_scheduler.CosineAnnealingLR(
    ot, T_max=MAX_EPOCHS, eta_min=1e-7)

# early stopping / checkpoint selection on validation accuracy
best = 0.0; pc = 0
txt_tl = []; txt_va = []

for ep in range(MAX_EPOCHS):
    txt_m.train(); tl = 0
    for x, l in tr_dl:
        ot.zero_grad()
        # Cast inputs to float32 to match model
        ids  = x['input_ids'].to(DEVICE)
        mask = x['attention_mask'].to(DEVICE)
        loss = mloss(txt_m(ids, mask), l)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(txt_m.parameters(), 1.0)
        ot.step()
        tl += loss.item()
    st.step()
    avg = tl / len(tr_dl)
    _, va, _, _ = eval_txt(txt_m, vl_dl)
    txt_tl.append(avg); txt_va.append(va)
    print(f"  Epoch {ep+1:2d}/{MAX_EPOCHS} | "
          f"Loss: {avg:.4f} | Val: {va*100:.2f}%")
          
    # keep the checkpoint with best VALIDATION accuracy
    if va > best:
        best = va; pc = 0
        torch.save(txt_m.state_dict(),
                   f"{OUTPUT_DIR}/txt_best.pth")
        print(f"  ✓ Best saved ({va*100:.2f}%)")
    else:
        pc += 1
        if pc >= PATIENCE:
            print(f"  Early stop at epoch {ep+1}")
            break

print("Text model training complete.")

# ── Test evaluation ───────────────────────────────────────────
txt_m.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/txt_best.pth"))

txt_accs, txt_overall, txt_preds, txt_truths = \
    eval_txt(txt_m, ts_dl)

print(f"\nText-Only TEST Accuracy: {txt_overall*100:.2f}%")
print("Per-attribute:")
for a, acc in txt_accs.items():
    print(f"  {a:15s}: {acc*100:.2f}%")

## 8. Train multimodal gated-fusion model (initialised from the two unimodal checkpoints)

In [ ]:
print("="*50)
print("TRAINING MULTIMODAL GATED FUSION MODEL")
print("="*50)

# Clear everything first
try:
    del img_m, txt_m, mm_m
except: pass
torch.cuda.empty_cache()
import gc; gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**2:.0f} MB")

tr_dl = DataLoader(MMDS(train_df, tokenizer, train_tfm),
                   BATCH_MULTI, shuffle=True,
                   num_workers=2, pin_memory=True)
vl_dl = DataLoader(MMDS(val_df, tokenizer, val_tfm),
                   BATCH_MULTI, shuffle=False,
                   num_workers=2, pin_memory=True)
ts_dl = DataLoader(MMDS(test_df, tokenizer, val_tfm),
                   BATCH_MULTI, shuffle=False,
                   num_workers=2, pin_memory=True)

mm_m = MultimodalModel(num_classes).to(DEVICE)

# Enable gradient checkpointing to save GPU memory
mm_m.enc.gradient_checkpointing_enable()

# Transfer pretrained weights
isd = torch.load(f"{OUTPUT_DIR}/img_best.pth",
                 map_location=DEVICE)
r = mm_m.backbone.load_state_dict(
    {k[9:]: v for k,v in isd.items()
     if k.startswith('backbone.')}, strict=False)
print("backbone transfer  missing:", r.missing_keys, "unexpected:", r.unexpected_keys)

tsd = torch.load(f"{OUTPUT_DIR}/txt_best.pth",
                 map_location=DEVICE)
r = mm_m.enc.load_state_dict(
    {k[4:]: v for k,v in tsd.items()
     if k.startswith('enc.')}, strict=False)
print("encoder transfer   missing:", r.missing_keys, "unexpected:", r.unexpected_keys)

# Force float32
mm_m = mm_m.float()
print("Pretrained weights transferred.")
print(f"GPU free after model load: "
      f"{torch.cuda.mem_get_info()[0]/1024**2:.0f} MB")

om = optim.AdamW([
    {'params': mm_m.backbone.parameters(), 'lr': LR_BACKBONE},
    {'params': mm_m.enc.parameters(),      'lr': LR_ENCODER_MM},
    {'params': mm_m.ip.parameters(),       'lr': LR_HEAD},
    {'params': mm_m.tp.parameters(),       'lr': LR_HEAD},
    {'params': mm_m.gate.parameters(),     'lr': LR_HEAD},
    {'params': mm_m.heads.parameters(),    'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

sm = optim.lr_scheduler.CosineAnnealingLR(
    om, T_max=MAX_EPOCHS, eta_min=1e-7)

# early stopping / checkpoint selection on validation accuracy
best = 0.0; pc = 0
mm_tl = []; mm_va = []

for ep in range(MAX_EPOCHS):
    mm_m.train(); tl = 0
    for img, x, l in tr_dl:
        om.zero_grad()
        with torch.cuda.amp.autocast(dtype=torch.float16):
            o, _ = mm_m(img.to(DEVICE).float(),
                        x['input_ids'].to(DEVICE),
                        x['attention_mask'].to(DEVICE))
            loss = mloss(o, l)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            mm_m.parameters(), 1.0)
        om.step()
        tl += loss.item()
    sm.step()
    avg = tl / len(tr_dl)
    _, va, _, _, _ = eval_mm(mm_m, vl_dl)
    mm_tl.append(avg); mm_va.append(va)
    print(f"  Epoch {ep+1:2d}/{MAX_EPOCHS} | "
          f"Loss: {avg:.4f} | Val: {va*100:.2f}%")

    # keep the checkpoint with best VALIDATION accuracy
    if va > best:
        best = va; pc = 0
        torch.save(mm_m.state_dict(),
                   f"{OUTPUT_DIR}/mm_best.pth")
        print(f"  ✓ Best saved ({va*100:.2f}%)")
    else:
        pc += 1
        if pc >= PATIENCE:
            print(f"  Early stop at epoch {ep+1}")
            break

print("Multimodal model training complete.")

# ── Test evaluation ───────────────────────────────────────────
mm_m.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/mm_best.pth"))

mm_accs, mm_overall, mm_alphas, mm_preds, mm_truths = \
    eval_mm(mm_m, ts_dl)

print(f"\nMultimodal TEST Accuracy: {mm_overall*100:.2f}%")
print("Per-attribute:")
for a, acc in mm_accs.items():
    print(f"  {a:15s}: {acc*100:.2f}%")

## 9. Reload best checkpoints and collect test predictions (with category ids)

In [ ]:
ts_img = DataLoader(ImgDS(test_df, val_tfm), BATCH_IMAGE, shuffle=False, num_workers=2)
ts_txt = DataLoader(TxtDS(test_df, tokenizer), BATCH_TEXT,  shuffle=False, num_workers=2)
ts_mm  = DataLoader(MMDS(test_df, tokenizer, val_tfm), BATCH_MULTI, shuffle=False, num_workers=2)

img_m = ImageModel(num_classes).to(DEVICE)
img_m.load_state_dict(torch.load(f"{OUTPUT_DIR}/img_best.pth", map_location=DEVICE)); img_m.eval()

txt_m = TextModel(num_classes).to(DEVICE)
txt_m.load_state_dict(torch.load(f"{OUTPUT_DIR}/txt_best.pth", map_location=DEVICE)); txt_m = txt_m.float().eval()

mm_m = MultimodalModel(num_classes).to(DEVICE)
mm_m.load_state_dict(torch.load(f"{OUTPUT_DIR}/mm_best.pth", map_location=DEVICE)); mm_m = mm_m.float().eval()

def collect(model, dl, kind):
    P = {a: [] for a in TARGET_ATTRIBUTES}; T = {a: [] for a in TARGET_ATTRIBUTES}; C = []; A = []
    with torch.no_grad():
        for batch in dl:
            if kind == 'img':
                x, l = batch; o = model(x.to(DEVICE))
            elif kind == 'txt':
                x, l = batch; o = model(x['input_ids'].to(DEVICE), x['attention_mask'].to(DEVICE))
            else:
                img, x, l = batch
                o, al = model(img.to(DEVICE).float(), x['input_ids'].to(DEVICE), x['attention_mask'].to(DEVICE))
                A += al.squeeze(1).cpu().tolist()
            C += l['category'].tolist()
            for a in TARGET_ATTRIBUTES:
                P[a] += o[a].argmax(1).cpu().tolist(); T[a] += l[a].tolist()
    return P, T, C, A

img_P, img_T, img_C, _         = collect(img_m, ts_img, 'img')
txt_P, txt_T, txt_C, _         = collect(txt_m, ts_txt, 'txt')
mm_P,  mm_T,  mm_C, mm_alphas  = collect(mm_m,  ts_mm,  'mm')
print("Test predictions collected.")

## 10. Test metrics (category-filtered; overall = unweighted mean of the 13 per-attribute accuracies)

In [ ]:
def wilson_ci(correct, total, z=1.96):
    p = correct / total
    c = (p + z**2/(2*total)) / (1 + z**2/total)
    m = (z * (p*(1-p)/total + z**2/(4*total**2))**0.5) / (1 + z**2/total)
    return (c-m)*100, (c+m)*100

def build_metrics(P, T, C, name):
    res, accs = {}, []
    for a in TARGET_ATTRIBUTES:
        fp, ft = filter_by_category(P[a], T[a], C, a)
        n = len(ft)
        acc = accuracy_score(ft, fp)
        accs.append(acc)                      # keep UNROUNDED for the overall mean
        res[a] = {
            'n': n, 'accuracy': acc*100,
            'weighted_precision': precision_score(ft, fp, average='weighted', zero_division=0)*100,
            'weighted_recall':    recall_score(ft, fp, average='weighted', zero_division=0)*100,
            'weighted_f1':        f1_score(ft, fp, average='weighted', zero_division=0)*100,
            'macro_f1':           f1_score(ft, fp, average='macro', zero_division=0)*100,
        }
    overall = float(np.mean(accs))
    lo, hi = wilson_ci(overall * len(test_df), len(test_df))   # Wilson CI with n = test-set size
    res['_overall'] = {'accuracy': overall*100, 'ci_lower': lo, 'ci_upper': hi}
    print(f"{name:12s} overall {overall*100:.2f}%  95% Wilson CI [{lo:.2f}, {hi:.2f}]")
    return res

metrics = {'image_only': build_metrics(img_P, img_T, img_C, 'Image-only'),
           'text_only':  build_metrics(txt_P, txt_T, txt_C, 'Text-only'),
           'multimodal': build_metrics(mm_P,  mm_T,  mm_C,  'Multimodal')}

print(f"\n{'attribute':14s}{'N':>6}{'image':>9}{'text':>9}{'MM':>9}{'MM-text':>9}{'W-F1':>8}{'M-F1':>8}")
for a in TARGET_ATTRIBUTES:
    i, t, m = metrics['image_only'][a], metrics['text_only'][a], metrics['multimodal'][a]
    print(f"{a:14s}{m['n']:>6}{i['accuracy']:>9.2f}{t['accuracy']:>9.2f}{m['accuracy']:>9.2f}"
          f"{m['accuracy']-t['accuracy']:>+9.2f}{m['weighted_f1']:>8.2f}{m['macro_f1']:>8.2f}")

with open(f"{OUTPUT_DIR}/metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)

## 11. McNemar test, text-only vs multimodal (per attribute)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(true, pred_a, pred_b, name_a, name_b, attr):
    # Build 2x2 contingency table: correct/incorrect for model A vs B
    correct_a = np.array([t==p for t,p in zip(true, pred_a)])
    correct_b = np.array([t==p for t,p in zip(true, pred_b)])
    
    n01 = np.sum((~correct_a) & correct_b)   # A wrong, B right
    n10 = np.sum(correct_a & (~correct_b))   # A right, B wrong
    
    table = [[0, n01], [n10, 0]]
    result = mcnemar(table, exact=(n01+n10 < 25), correction=True)
    print(f"  {attr:15s} | {name_a} vs {name_b}: statistic={result.statistic:.2f}, p={result.pvalue:.6f}")
    return result.pvalue

print("="*60)
print("MCNEMAR TEST: Text-Only vs Multimodal (per attribute)")
print("="*60)
mcnemar_results = {}
for a in TARGET_ATTRIBUTES:
    fp_txt, ft = filter_by_category(txt_P[a], txt_T[a], txt_C, a)
    fp_mm,  _  = filter_by_category(mm_P[a],  mm_T[a],  mm_C,  a)
    # ensure same length/order — both filtered from same category mask logic
    p = mcnemar_test(ft, fp_txt, fp_mm, "Text-Only", "Multimodal", a)
    mcnemar_results[a] = {'p_value': float(p), 'significant': bool(p < 0.05)}

with open(f"{OUTPUT_DIR}/mcnemar_results.json", 'w') as f:
    json.dump(mcnemar_results, f, indent=2)
print('Saved mcnemar_results.json')

## 12. Gate analysis (overall, per attribute-applicability group, per category)

In [ ]:
A = np.array(mm_alphas)
print(f"mean {A.mean():.4f} | std {A.std():.4f} | min {A.min():.4f} | max {A.max():.4f}")
print(f"image-dominant (alpha>0.5): {(A>0.5).mean()*100:.1f}%   text-dominant (<0.5): {(A<0.5).mean()*100:.1f}%")

C_arr = np.array(mm_C)
print("\nMean alpha per attribute (category-applicable samples only):")
for a in TARGET_ATTRIBUTES:
    if a in ATTR_VALID_INDICES:
        mask = np.isin(C_arr, list(ATTR_VALID_INDICES[a]))
    else:
        mask = np.ones(len(A), dtype=bool)
    print(f"  {a:14s} {A[mask].mean():.4f}  N={int(mask.sum())}")

cat_names = label_encoders['category'].classes_
print("\nMean alpha per category:")
for ci in np.unique(C_arr):
    print(f"  {cat_names[ci]:20s} {A[C_arr == ci].mean():.4f}")

## 13. Robustness: degraded text (checkpoint is not retrained)
`Empty Text` replaces every description with the literal placeholder `[NO DESCRIPTION]`.

In [ ]:
random.seed(SEED); np.random.seed(SEED)

def rob_eval(cond, fn):
    tmp = test_df.copy()
    tmp['masked_text'] = tmp['masked_text'].apply(fn)
    dl_mm  = DataLoader(MMDS(tmp, tokenizer, val_tfm), BATCH_MULTI, shuffle=False, num_workers=2, pin_memory=True)
    dl_txt = DataLoader(TxtDS(tmp, tokenizer), BATCH_TEXT, shuffle=False, num_workers=2, pin_memory=True)
    _, mma, al, _, _ = eval_mm(mm_m, dl_mm)
    _, txa, _, _     = eval_txt(txt_m, dl_txt)
    al = np.array(al)
    print(f"  {cond:15s} text {txa*100:6.2f}%  MM {mma*100:6.2f}%  gain {(mma-txa)*100:+6.2f}   "
          f"gate mean {al.mean():.4f}, image-dominant {(al>0.5).mean()*100:.1f}%")
    return {'txt': txa*100, 'mm': mma*100, 'gain': (mma-txa)*100,
            'gate_mean': float(al.mean()), 'img_dom_pct': float((al>0.5).mean()*100)}

robustness = {
    'Normal Text':   rob_eval('Normal Text', lambda t: t),
    '50% Word Drop': rob_eval('50% Word Drop', lambda t: ' '.join([w for w in t.split() if random.random() > 0.5]) if t.split() else t),
    'Word Shuffle':  rob_eval('Word Shuffle', lambda t: ' '.join(random.sample(t.split(), len(t.split()))) if t.split() else t),
    'Empty Text':    rob_eval('Empty Text', lambda t: '[NO DESCRIPTION]'),
}
with open(f"{OUTPUT_DIR}/robustness.json", 'w') as f:
    json.dump(robustness, f, indent=2)

## 14. Fixed-weight fusion ablation (inference-time override of the learned gate, no retraining)

In [ ]:
def eval_mm_fixed_alpha(model, dl, alpha_const):
    """Same forward pass as MultimodalModel, but alpha is replaced by a constant."""
    model.eval()
    P = {a: [] for a in TARGET_ATTRIBUTES}; T = {a: [] for a in TARGET_ATTRIBUTES}; C = []
    with torch.no_grad():
        for img, x, l in dl:
            iv = model.ip(model.backbone(img.to(DEVICE).float())).float()
            tv = model.tp(model.enc(input_ids=x['input_ids'].to(DEVICE),
                                    attention_mask=x['attention_mask'].to(DEVICE)
                                    ).last_hidden_state[:, 0, :]).float()
            f = model.drop(alpha_const * iv + (1 - alpha_const) * tv)
            C += l['category'].tolist()
            for a in TARGET_ATTRIBUTES:
                P[a] += model.heads[a](f).argmax(1).cpu().tolist(); T[a] += l[a].tolist()
    accs = []
    for a in TARGET_ATTRIBUTES:
        fp, ft = filter_by_category(P[a], T[a], C, a)
        accs.append(accuracy_score(ft, fp))
    return float(np.mean(accs)) * 100

empty_df = test_df.copy(); empty_df['masked_text'] = '[NO DESCRIPTION]'
dl_normal = DataLoader(MMDS(test_df,  tokenizer, val_tfm), BATCH_MULTI, shuffle=False, num_workers=2)
dl_empty  = DataLoader(MMDS(empty_df, tokenizer, val_tfm), BATCH_MULTI, shuffle=False, num_workers=2)

print(f"{'config':28s}{'normal text':>14s}{'empty text':>14s}")
for name, al in [('fixed alpha = 0.5', 0.5), ('fixed alpha = 1.0 (image path)', 1.0), ('fixed alpha = 0.0 (text path)', 0.0)]:
    n = eval_mm_fixed_alpha(mm_m, dl_normal, al)
    e = eval_mm_fixed_alpha(mm_m, dl_empty, al) if al == 0.5 else float('nan')
    print(f"{name:28s}{n:>13.2f}%{e:>13.2f}%")
print(f"{'learned gate':28s}{metrics['multimodal']['_overall']['accuracy']:>13.2f}%{robustness['Empty Text']['mm']:>13.2f}%")

## 15. Efficiency (single-sample latency, peak GPU memory)
Peak memory is measured with the three models resident in the same session.

In [ ]:
# ══════════════════════════════════════════════════════
# EFFICIENCY METRICS: Inference Time + GPU Memory
# ══════════════════════════════════════════════════════
import time

img_m.eval(); txt_m.eval(); mm_m.eval()

sample_img  = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
sample_txt  = tokenizer("mens casual round neck cotton t-shirt",
                         return_tensors='pt', max_length=MAX_TOKEN_LEN,
                         padding='max_length', truncation=True)
sample_ids  = sample_txt['input_ids'].to(DEVICE)
sample_mask = sample_txt['attention_mask'].to(DEVICE)

def measure_inference(model, mtype, n_runs=200, n_warmup=20):
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            if mtype == 'img': model(sample_img)
            elif mtype == 'txt': model(sample_ids, sample_mask)
            else: model(sample_img, sample_ids, sample_mask)
    torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            if mtype == 'img': model(sample_img)
            elif mtype == 'txt': model(sample_ids, sample_mask)
            else: model(sample_img, sample_ids, sample_mask)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return round(sum(times) / len(times), 2)

def measure_peak_memory(model, mtype):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        if mtype == 'img': model(sample_img)
        elif mtype == 'txt': model(sample_ids, sample_mask)
        else: model(sample_img, sample_ids, sample_mask)
    torch.cuda.synchronize()
    return round(torch.cuda.max_memory_allocated() / 1024**2, 1)

print("Measuring inference time (200 runs each)...")
img_ms = measure_inference(img_m, 'img')
txt_ms = measure_inference(txt_m, 'txt')
mm_ms  = measure_inference(mm_m,  'mm')
print(f"Image-Only:  {img_ms} ms")
print(f"Text-Only:   {txt_ms} ms")
print(f"Multimodal:  {mm_ms} ms")

print("\nMeasuring peak GPU memory...")
img_mb = measure_peak_memory(img_m, 'img')
txt_mb = measure_peak_memory(txt_m, 'txt')
mm_mb  = measure_peak_memory(mm_m,  'mm')
print(f"Image-Only:  {img_mb} MB")
print(f"Text-Only:   {txt_mb} MB")
print(f"Multimodal:  {mm_mb} MB")

throughput = round(1000/mm_ms, 1)
time_10k   = round(10000/(1000/mm_ms)/60, 1)
print(f"\nThroughput (multimodal): {throughput} samples/sec")
print(f"Time for 10,000 products: {time_10k} minutes")

efficiency = {
    'inference_ms': {'image_only': img_ms, 'text_only': txt_ms, 'multimodal': mm_ms},
    'gpu_memory_mb': {'image_only': img_mb, 'text_only': txt_mb, 'multimodal': mm_mb},
    'throughput_per_sec': throughput,
}
with open(f"{OUTPUT_DIR}/efficiency_metrics.json", 'w') as f:
    json.dump(efficiency, f, indent=2)